# MLOps: CI/CD for AI

The [infrastructure notebook](/courses/llm-eng/12-ai-infra.html) gave us a running service: a containerised filing analyser deployed on Kubernetes, provisioned with Terraform, and wired into a GitHub Actions build pipeline. What it does not give us is a closed loop — a systematic way to detect when the model degrades, retrain it, validate the new version, and promote it to production without manual intervention. That loop is the subject of MLOps. In this deep dive we implement the full cycle: a model registry backed by SQLite, embedding and output quality drift detectors, shadow and canary deployment routers, an automated retraining trigger, and a reference GitHub Actions workflow that ties everything together in the context of a financial compliance AI system.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## MLOps vs. Traditional DevOps

Traditional DevOps manages one axis of change: **code**. A new commit triggers tests, builds an artifact, and deploys it. ML systems have three axes:

1. **Code** — the training pipeline, inference service, prompt templates.
2. **Data** — the training corpus, retrieval index, fine-tuning examples. A regulatory update (say, a new Basel IV capital rule) can silently invalidate a compliance classifier trained on older SEC filings even without touching a single line of code.
3. **Model** — the weights, adapter layers, or the upstream foundation model itself. A vendor silently rolling `gpt-4o` to a new checkpoint is a model change you did not initiate.

This three-axis structure generates **ML-specific technical debt** that Sculley et al. famously catalogued: hidden feedback loops (a model's predictions influence future training data), undeclared consumers (other services depend on a model's output schema), and pipeline jungles (ad-hoc scripts that no one is sure are still needed). In a regulated financial institution, these are not just engineering inconveniences — they are audit findings. A compliance model must have a traceable version, a recorded training date, the hash of the prompt template it uses, and the metrics on which it was approved for production.

<br>

**MLOps maturity model.** Google's MLOps whitepaper defines three maturity levels:

| Level | Label | What is automated |
|:---:|:---|:---|
| 0 | Manual | Nothing. Data scientists export a pickle and email it to an engineer. |
| 1 | Automated training | The training pipeline runs on a schedule or trigger, but model validation and deployment are still manual. |
| 2 | Automated CI/CD for ML | The full loop — trigger, retrain, validate, shadow-deploy, canary, promote — is automated. Human approval is reserved for production promotion only. |

Most financial AI teams operate at level 1. This deep dive builds the components for level 2.

## Model Registry

A **model registry** is the single source of truth for model versions and their lifecycle stage. It stores not just weights but the metadata needed for audit: training date, evaluation metrics, the hash of the prompt template, and the base model name. We implement a lightweight registry backed by SQLite — sufficient for a single-team system and easy to replace with MLflow or SageMaker Model Registry when the team scales.

Each version can be in one of three lifecycle stages: `staging` (freshly registered, under evaluation), `production` (serving live traffic), or `archived` (retired). The invariant is that at most one version per model name is in `production` at any time.

Defining the `ModelRegistry` class:

In [ ]:
import sqlite3
import hashlib
import datetime
from dataclasses import dataclass, field, asdict


@dataclass
class ModelVersion:
    name: str
    version: str
    base_model: str
    prompt_hash: str
    metrics: dict
    artifact_path: str
    stage: str = "staging"                                # <1>
    training_date: str = field(
        default_factory=lambda: datetime.datetime.utcnow().isoformat()
    )


class ModelRegistry:
    """Lightweight SQLite-backed model registry with lifecycle management."""

    _DDL = """
    CREATE TABLE IF NOT EXISTS models (
        id            INTEGER PRIMARY KEY AUTOINCREMENT,
        name          TEXT    NOT NULL,
        version       TEXT    NOT NULL,
        base_model    TEXT    NOT NULL,
        prompt_hash   TEXT    NOT NULL,
        metrics       TEXT    NOT NULL,
        artifact_path TEXT    NOT NULL,
        stage         TEXT    NOT NULL DEFAULT 'staging',
        training_date TEXT    NOT NULL,
        UNIQUE(name, version)
    );
    """

    def __init__(self, db_path: str = ":memory:"):
        self._db_path = db_path
        self._conn = sqlite3.connect(db_path, check_same_thread=False)  # <2>
        self._conn.row_factory = sqlite3.Row
        self._conn.executescript(self._DDL)
        self._conn.commit()

    def register(self, mv: ModelVersion) -> int:
        """Insert a new model version at stage='staging'. Returns row id."""
        cur = self._conn.execute(
            """INSERT INTO models
               (name, version, base_model, prompt_hash, metrics,
                artifact_path, stage, training_date)
               VALUES (?,?,?,?,?,?,?,?)""",
            (
                mv.name, mv.version, mv.base_model, mv.prompt_hash,
                json.dumps(mv.metrics), mv.artifact_path,
                mv.stage, mv.training_date,
            ),
        )
        self._conn.commit()
        return cur.lastrowid

    def promote(self, name: str, version: str) -> None:
        """Promote version to production; archive the previous production model."""
        with self._conn:                                  # <3>
            self._conn.execute(
                "UPDATE models SET stage='archived' WHERE name=? AND stage='production'",
                (name,),
            )
            self._conn.execute(
                "UPDATE models SET stage='production' WHERE name=? AND version=?",
                (name, version),
            )

    def rollback(self, name: str) -> Optional[str]:
        """Demote current production to staging; return the version rolled back."""
        row = self._conn.execute(
            "SELECT version FROM models WHERE name=? AND stage='production'",
            (name,),
        ).fetchone()
        if row is None:
            return None
        version = row["version"]
        self._conn.execute(
            "UPDATE models SET stage='staging' WHERE name=? AND version=?",
            (name, version),
        )
        self._conn.commit()
        return version

    def get_production(self, name: str) -> Optional[ModelVersion]:
        """Return the current production ModelVersion, or None."""
        row = self._conn.execute(
            "SELECT * FROM models WHERE name=? AND stage='production'",
            (name,),
        ).fetchone()
        if row is None:
            return None
        return ModelVersion(
            name=row["name"], version=row["version"],
            base_model=row["base_model"], prompt_hash=row["prompt_hash"],
            metrics=json.loads(row["metrics"]),
            artifact_path=row["artifact_path"], stage=row["stage"],
            training_date=row["training_date"],
        )

    def list_versions(self, name: str) -> list[dict]:
        """Return all versions for a model name, newest first."""
        rows = self._conn.execute(
            "SELECT * FROM models WHERE name=? ORDER BY id DESC", (name,)
        ).fetchall()
        return [{k: row[k] for k in row.keys()} for row in rows]  # <4>

1. New versions enter at `staging` by default — they are never automatically live.
2. `check_same_thread=False` allows the registry to be called from async contexts (e.g., FastAPI route handlers) without creating a new connection per request.
3. Using `with self._conn` as a context manager wraps the two UPDATE statements in a single transaction, so there is never a moment where zero versions are in production (important for a live service).
4. `sqlite3.Row` acts like both a tuple and a dict, so `row.keys()` gives column names.

Exercising the full lifecycle — register, promote, rollback:

In [ ]:
def prompt_hash(template: str) -> str:
    return hashlib.sha256(template.encode()).hexdigest()[:12]


PROMPT_V1 = "You are a compliance classifier. Classify the following SEC filing excerpt."
PROMPT_V2 = "You are a financial compliance classifier. Given an SEC filing excerpt, output one of: risk_factors | mda | capital_liquidity | guidance."

registry = ModelRegistry(db_path=":memory:")

v1 = ModelVersion(
    name="compliance-classifier",
    version="1.0.0",
    base_model="gpt-4o-mini",
    prompt_hash=prompt_hash(PROMPT_V1),
    metrics={"accuracy": 0.84, "f1": 0.82},
    artifact_path="s3://models/compliance/v1.0.0/",
)

v2 = ModelVersion(
    name="compliance-classifier",
    version="1.1.0",
    base_model="gpt-4o-mini",
    prompt_hash=prompt_hash(PROMPT_V2),
    metrics={"accuracy": 0.91, "f1": 0.90},
    artifact_path="s3://models/compliance/v1.1.0/",
)

registry.register(v1)
registry.register(v2)

# Promote v1 first, then upgrade to v2
registry.promote("compliance-classifier", "1.0.0")
prod = registry.get_production("compliance-classifier")
print(f"Production after first promote : {prod.version}  (metrics={prod.metrics})")

registry.promote("compliance-classifier", "1.1.0")
prod = registry.get_production("compliance-classifier")
print(f"Production after second promote: {prod.version}  (metrics={prod.metrics})")

# Simulate a bad deploy — rollback
rolled = registry.rollback("compliance-classifier")
print(f"Rolled back version           : {rolled}")
print(f"Production after rollback     : {registry.get_production('compliance-classifier')}")

print("\nAll versions:")
for row in registry.list_versions("compliance-classifier"):
    print(f"  {row['version']}  stage={row['stage']}  prompt_hash={row['prompt_hash']}")

:::{.callout-note}
In a regulated environment the `promote()` call should require a human approval step — a GitHub Actions environment protection rule, an AWS Systems Manager Change Manager approval, or a simple database flag set by an authorized user. The registry itself should be append-only (no DELETEs) so that a complete audit trail of every version's lifecycle is preserved.

:::

## Embedding Distribution Drift

The first class of drift we care about is **covariate shift**: the distribution of incoming queries changes, even if the model itself has not. In a financial AI system this happens when, for example, users start asking about a newly enacted regulation that does not appear in the training corpus. If the retrieval index was built on older filings, the shifted queries will land in a sparse region of embedding space and produce poor retrievals — but the model will not raise an error. The symptom is silent quality degradation.

We detect this by comparing the distribution of recent query embeddings against a reference distribution (collected during a known-good period) using **Maximum Mean Discrepancy** (MMD). Given two distributions $P$ and $Q$ with samples $\{x_i\}$ and $\{y_j\}$, the unbiased estimator of $\text{MMD}^2$ under an RBF kernel $k(x, y) = \exp(-\|x - y\|^2 / (2\sigma^2))$ is:

$$\text{MMD}^2(P, Q) = \mathbb{E}_{x,x' \sim P}[k(x,x')] - 2\,\mathbb{E}_{x \sim P,\, y \sim Q}[k(x,y)] + \mathbb{E}_{y,y' \sim Q}[k(y,y')]$$ {#eq-mmd}

When $P = Q$, $\text{MMD}^2 = 0.$ A value above a calibrated threshold indicates the two samples are drawn from different distributions. The RBF kernel bandwidth $\sigma$ is set via the **median heuristic**: $\sigma^2 = \text{median}(\|x_i - x_j\|^2) / 2.$

Implementing the `EmbeddingDriftDetector`:

In [ ]:
class EmbeddingDriftDetector:
    """MMD-based drift detector for query embedding distributions."""

    def __init__(self, reference: np.ndarray, threshold: float = 0.05):
        self._ref = reference                             # <1>
        self.threshold = threshold
        self._sigma2 = self._median_bandwidth(reference)

    @staticmethod
    def _median_bandwidth(X: np.ndarray) -> float:
        """Median heuristic for RBF kernel bandwidth."""
        n = min(len(X), 500)                              # <2>
        X_sub = X[:n]
        dists_sq = np.sum(
            (X_sub[:, None, :] - X_sub[None, :, :]) ** 2, axis=-1
        )
        return float(np.median(dists_sq[dists_sq > 0])) / 2.0

    def _rbf_kernel(self, A: np.ndarray, B: np.ndarray) -> np.ndarray:
        """RBF kernel matrix K[i,j] = exp(-||A_i - B_j||^2 / (2*sigma^2))."""
        dists_sq = np.sum(
            (A[:, None, :] - B[None, :, :]) ** 2, axis=-1
        )
        return np.exp(-dists_sq / (2.0 * self._sigma2))

    def mmd2(self, window: np.ndarray) -> float:
        """Compute unbiased MMD^2 between reference and a query window."""
        X, Y = self._ref, window
        Kxx = self._rbf_kernel(X, X)                     # <3>
        Kyy = self._rbf_kernel(Y, Y)
        Kxy = self._rbf_kernel(X, Y)
        n, m = len(X), len(Y)
        # Unbiased: zero out diagonal for same-sample terms
        np.fill_diagonal(Kxx, 0.0)
        np.fill_diagonal(Kyy, 0.0)
        term_xx = Kxx.sum() / (n * (n - 1))
        term_yy = Kyy.sum() / (m * (m - 1))
        term_xy = Kxy.mean()
        return float(term_xx - 2 * term_xy + term_yy)

    def check(self, window: np.ndarray) -> dict:
        """Return dict with mmd2 score and alert flag."""
        score = self.mmd2(window)
        return {"mmd2": score, "alert": score > self.threshold}  # <4>

1. The reference distribution is a collection of embeddings gathered during a known-good period, e.g., the first two weeks after a model launch.
2. Computing the full pairwise distance matrix is $O(n^2 d)$. We cap at 500 samples for the bandwidth estimate to keep it tractable on high-dimensional embeddings.
3. $K_{XX}$ is the kernel matrix between all reference pairs, $K_{YY}$ between all window pairs, and $K_{XY}$ cross-terms. The unbiased estimator zeros out the diagonal of the same-sample matrices since $k(x, x) = 1$ always and would bias the estimate upward.
4. The `alert` flag is what a monitoring system polls. The threshold of $0.05$ is a reasonable starting point but should be calibrated on held-out no-drift windows via permutation testing.

Simulating no-drift and drifted query windows to verify the detector fires correctly:

In [ ]:
rng = np.random.default_rng(42)
D = 32  # Reduced embedding dim for simulation

# Reference: queries about Basel III capital rules
reference_embeddings = rng.normal(loc=0.0, scale=1.0, size=(200, D))

# No-drift window: same distribution
window_nodrift = rng.normal(loc=0.0, scale=1.0, size=(100, D))

# Drifted window: users start asking about new crypto-asset regulations
# — embedding centroid shifted by 3 standard deviations
window_drift = rng.normal(loc=3.0, scale=1.0, size=(100, D))

detector = EmbeddingDriftDetector(reference=reference_embeddings, threshold=0.05)

result_nodrift = detector.check(window_nodrift)
result_drift   = detector.check(window_drift)

print(f"No-drift window  — MMD²={result_nodrift['mmd2']:.4f}  alert={result_nodrift['alert']}")
print(f"Drifted window   — MMD²={result_drift['mmd2']:.4f}  alert={result_drift['alert']}")

Plotting MMD² over a rolling series of windows as the shift gradually increases:

In [ ]:
#| code-fold: true
%config InlineBackend.figure_formats = ['svg']
import matplotlib.pyplot as plt

shifts = np.linspace(0, 4, 30)
mmd_scores = []
for shift in shifts:
    w = rng.normal(loc=shift, scale=1.0, size=(100, D))
    mmd_scores.append(detector.mmd2(w))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(shifts, mmd_scores, color="#2b6cb0", linewidth=2, label="MMD²")
ax.axhline(detector.threshold, color="#e53e3e", linestyle="--", linewidth=1.5, label="threshold")
ax.fill_between(
    shifts, mmd_scores, detector.threshold,
    where=[s > detector.threshold for s in mmd_scores],
    alpha=0.15, color="#e53e3e",
)
ax.set_xlabel("Distribution shift (σ units)", fontsize=11)
ax.set_ylabel("MMD²", fontsize=11)
ax.set_title("Embedding drift detection", fontsize=12)
ax.legend(fontsize=10)
ax.grid(linestyle="dotted", alpha=0.6)
plt.tight_layout();

**Figure.** MMD² as the query distribution shifts away from the reference. The detector fires (shaded region) once the mean shift exceeds roughly 1.5σ. Below that, natural within-distribution variability keeps MMD² under the threshold.

## Output Quality Drift

Embedding drift is a leading indicator. The lagging indicator — and the one regulators care about — is **output quality**: are the model's answers still faithful to the retrieved context, and are they still relevant to the question? We track two RAGAS-style scores per query: *faithfulness* (does the answer contain only information supported by the retrieved chunks?) and *answer relevancy* (does the answer address what was asked?).

Rather than detecting a single event, we want to know when the system has been consistently underperforming. We use an **Exponentially Weighted Moving Average** (EWMA) with span $s = 20$, which corresponds to a decay factor $\alpha = 2/(s+1) = 2/21 \approx 0.095.$ The EWMA at step $t$ is:

$$\mu_t = \alpha \cdot x_t + (1-\alpha) \cdot \mu_{t-1}$$

An alert fires when $\mu_t$ drops below a minimum acceptable score — we use $0.70$ for both faithfulness and relevancy, consistent with the RAGAS thresholds from [notebook 07](/courses/llm-eng/07-rag-pipeline.html).

Implementing `QualityDriftMonitor`:

In [ ]:
from collections import deque


class QualityDriftMonitor:
    """EWMA-based monitor for faithfulness and relevancy score degradation."""

    def __init__(
        self,
        span: int = 20,
        threshold: float = 0.70,
        history_len: int = 500,
    ):
        self._alpha = 2.0 / (span + 1)                   # <1>
        self.threshold = threshold
        self._ewma_faith: Optional[float] = None
        self._ewma_relev: Optional[float] = None
        self._history = deque(maxlen=history_len)         # <2>

    def update(self, faithfulness: float, relevancy: float) -> dict:
        """Record a new observation and return current EWMA state and alert."""
        if self._ewma_faith is None:                      # <3>
            self._ewma_faith = faithfulness
            self._ewma_relev = relevancy
        else:
            self._ewma_faith = self._alpha * faithfulness + (1 - self._alpha) * self._ewma_faith
            self._ewma_relev = self._alpha * relevancy    + (1 - self._alpha) * self._ewma_relev

        state = {
            "ewma_faithfulness": round(self._ewma_faith, 4),
            "ewma_relevancy":    round(self._ewma_relev, 4),
            "alert": (
                self._ewma_faith < self.threshold or
                self._ewma_relev < self.threshold
            ),
        }
        self._history.append({"faithfulness": faithfulness, "relevancy": relevancy, **state})
        return state

    @property
    def current(self) -> dict:
        """Return the latest EWMA values without adding a new observation."""
        return {
            "ewma_faithfulness": round(self._ewma_faith or 0.0, 4),
            "ewma_relevancy":    round(self._ewma_relev or 0.0, 4),
        }

    def get_history(self) -> list[dict]:
        return list(self._history)

1. A span of 20 means the most recent observation carries weight $\alpha \approx 9.5\%$ and observations more than ~$3/\alpha \approx 31$ steps ago are nearly forgotten. This makes the EWMA responsive to trends without being noisy.
2. We keep a rolling history window for retrospective debugging — e.g., to identify which batch of queries triggered a degradation.
3. We initialise the EWMA with the first observation rather than zero to avoid the warm-up bias that would suppress early alerts.

Simulating 150 queries: 80 healthy, then 70 degraded (e.g., a prompt template regression introduced by a bad deploy):

In [ ]:
monitor = QualityDriftMonitor(span=20, threshold=0.70)

rng2 = np.random.default_rng(7)

# Phase 1: healthy system (faithfulness ~0.87, relevancy ~0.85)
healthy_faith = rng2.normal(0.87, 0.05, 80).clip(0, 1)
healthy_relev = rng2.normal(0.85, 0.06, 80).clip(0, 1)

# Phase 2: degraded — prompt regression drops scores sharply
bad_faith = rng2.normal(0.60, 0.07, 70).clip(0, 1)
bad_relev = rng2.normal(0.58, 0.08, 70).clip(0, 1)

history = []
for f, r in zip(healthy_faith, healthy_relev):
    history.append(monitor.update(float(f), float(r)))
for f, r in zip(bad_faith, bad_relev):
    history.append(monitor.update(float(f), float(r)))

# Find first alert step
first_alert = next((i for i, h in enumerate(history) if h["alert"]), None)
print(f"First alert at step : {first_alert}")
print(f"EWMA at alert       : faithfulness={history[first_alert]['ewma_faithfulness']}  "
      f"relevancy={history[first_alert]['ewma_relevancy']}")

Plotting the EWMA traces with the alert region highlighted:

In [ ]:
#| code-fold: true
steps = list(range(len(history)))
ewma_f = [h["ewma_faithfulness"] for h in history]
ewma_r = [h["ewma_relevancy"]    for h in history]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(steps, ewma_f, color="#2b6cb0", linewidth=2, label="EWMA faithfulness")
ax.plot(steps, ewma_r, color="#38a169", linewidth=2, label="EWMA relevancy")
ax.axhline(monitor.threshold, color="#e53e3e", linestyle="--", linewidth=1.5, label="threshold")
ax.axvline(80, color="#718096", linestyle=":", linewidth=1.5, label="degradation starts")
if first_alert is not None:
    ax.axvline(first_alert, color="#e53e3e", linestyle="-", linewidth=1.5, alpha=0.7, label="alert fires")
ax.set_xlabel("Query step", fontsize=11)
ax.set_ylabel("EWMA score", fontsize=11)
ax.set_title("Output quality drift — EWMA monitor", fontsize=12)
ax.legend(fontsize=9, ncol=3)
ax.grid(linestyle="dotted", alpha=0.6)
ax.set_ylim(0.45, 1.0)
plt.tight_layout();

**Figure.** EWMA traces for faithfulness and relevancy across 150 simulated queries. The degradation is injected at step 80 (dotted vertical line). The alert fires approximately 8–12 steps later as the EWMA integrates enough evidence to cross the threshold — quickly enough to limit user exposure, but late enough to avoid false positives from a single anomalous query.

## Shadow Deployment

Before promoting a new model to production we need to understand how it behaves on real traffic — not on an eval set. **Shadow deployment** achieves this by routing every live request to both the current production model and the candidate model in parallel. Only the production response is returned to the caller; the shadow response is logged for offline comparison. Users are unaffected, latency is determined by the production model (shadow calls happen concurrently), and we accumulate a representative sample of paired responses for evaluation.

In the compliance context: suppose we want to test a new `gpt-4o` based classifier against the production `gpt-4o-mini` classifier. Shadow deployment lets us see how often the two models disagree on real regulatory queries before committing to a promotion.

Implementing `ShadowRouter`:

In [ ]:
import threading
from dataclasses import dataclass as dc, field as f_field


@dc
class ShadowRecord:
    query: str
    prod_response: str
    shadow_response: str
    agree: bool


class ShadowRouter:
    """
    Routes every request to both production and shadow models.
    Returns only the production response; logs both for offline eval.
    """

    def __init__(
        self,
        prod_model: str = "gpt-4o-mini",
        shadow_model: str = "gpt-4o",
    ):
        self._prod   = LLMClient(model=prod_model)
        self._shadow = LLMClient(model=shadow_model)
        self._log: list[ShadowRecord] = []
        self._lock = threading.Lock()                     # <1>

    def _call(self, client: LLMClient, messages: list, result: list) -> None:
        """Thread target: append response to result list."""
        result.append(client.complete(messages))

    def route(self, messages: list) -> str:
        """Send messages to both models concurrently; return prod response."""
        prod_result, shadow_result = [], []

        t_prod   = threading.Thread(target=self._call, args=(self._prod,   messages, prod_result))   # <2>
        t_shadow = threading.Thread(target=self._call, args=(self._shadow, messages, shadow_result))

        t_prod.start(); t_shadow.start()
        t_prod.join();  t_shadow.join()                   # <3>

        prod_resp   = prod_result[0]   if prod_result   else ""
        shadow_resp = shadow_result[0] if shadow_result else ""

        query = next((m["content"] for m in messages if m["role"] == "user"), "")
        agree = prod_resp.strip().lower() == shadow_resp.strip().lower()

        with self._lock:
            self._log.append(ShadowRecord(
                query=query,
                prod_response=prod_resp,
                shadow_response=shadow_resp,
                agree=agree,
            ))

        return prod_resp                                  # <4>

    def agreement_rate(self) -> float:
        """Fraction of requests where prod and shadow agreed."""
        if not self._log:
            return float("nan")
        return sum(r.agree for r in self._log) / len(self._log)

    def get_log(self) -> list[ShadowRecord]:
        with self._lock:
            return list(self._log)

1. A `threading.Lock` protects the shared log from concurrent appends when the router is used in a multi-threaded server.
2. Both calls are launched in parallel threads so that shadow latency does not add to the user-facing response time. In an async service these would be `asyncio.gather` coroutines instead.
3. We wait for both threads to finish before appending the log record. In a production system the `t_shadow.join()` would be fire-and-forget (with a timeout) to avoid blocking on shadow failures.
4. Only the production response is returned. The caller has no visibility into the shadow model's response.

Running three shadow calls on compliance classification queries:

In [ ]:
SYSTEM_CLASSIFY = (
    "Classify the following SEC filing excerpt into exactly one category. "
    "Reply with only the category label: "
    "risk_factors | mda | capital_liquidity | guidance"
)

SHADOW_QUERIES = [
    "Our CET1 capital ratio was 14.8% at year-end, above the regulatory minimum of 4.5%.",
    "Net revenues for the fiscal year were $47.4 billion, an increase of 8% from prior year.",
    "We expect CET1 to remain in the 13.5–15.0% range through fiscal 2025.",
]

router = ShadowRouter(prod_model="gpt-4o-mini", shadow_model="gpt-4o")

for query in SHADOW_QUERIES:
    msgs = [
        {"role": "system", "content": SYSTEM_CLASSIFY},
        {"role": "user",   "content": query},
    ]
    prod_resp = router.route(msgs)
    print(f"prod={prod_resp!r:25s}  query={query[:55]}...")

print(f"\nAgreement rate: {router.agreement_rate():.0%}")
print("\nFull shadow log:")
for rec in router.get_log():
    marker = "✓" if rec.agree else "✗"
    print(f"  {marker}  prod={rec.prod_response!r:20s}  shadow={rec.shadow_response!r}")

:::{.callout-tip}
Agreement rate alone is not a sufficient promotion criterion. When the two models disagree, a human reviewer should inspect the disagreements to determine which model was correct. A candidate model that disagrees frequently but is *more often right* than production on disagreements is worth promoting; one that is merely *different* is not.

:::

## Canary Release

Shadow deployment validates correctness on real traffic but adds cost (two model calls per request). **Canary release** goes one step further: the new model serves a controlled fraction $p$ of live traffic and receives real user feedback — while limiting blast radius if it misbehaves. We start at $p = 0.05$ (5% canary), monitor error rates, and ramp $p$ upward if the canary's error rate stays below $2\times$ the baseline. If the canary exceeds the threshold at any point, we flip $p$ back to 0 and alert.

In practice, "error" in an LLM serving context means the model returned a response outside the allowed label set, raised an exception, or exceeded a latency budget. We track raw error counts per version.

Implementing `CanaryRouter`:

In [ ]:
import random


class CanaryRouter:
    """
    Routes p% of traffic to the canary model, (1-p)% to baseline.
    Auto-rolls back if canary error rate exceeds 2x baseline.
    """

    def __init__(
        self,
        baseline: LLMClient,
        canary: LLMClient,
        p: float = 0.05,
        rollback_multiplier: float = 2.0,
        valid_labels: set[str] | None = None,
    ):
        self._baseline = baseline
        self._canary   = canary
        self.p = p
        self._rollback_mult = rollback_multiplier
        self._valid_labels  = valid_labels
        self._stats = {
            "baseline": {"calls": 0, "errors": 0},
            "canary":   {"calls": 0, "errors": 0},
        }
        self._rolled_back = False
        self._alerts: list[str] = []

    def _is_error(self, response: str) -> bool:
        """True if response is outside the allowed label set."""
        if self._valid_labels is None:
            return False
        return response.strip().lower() not in self._valid_labels  # <1>

    def route(self, messages: list) -> dict:
        """Route a request; return response, version used, and alert state."""
        use_canary = (not self._rolled_back) and (random.random() < self.p)  # <2>
        version    = "canary" if use_canary else "baseline"
        client     = self._canary if use_canary else self._baseline

        try:
            response = client.complete(messages)
            is_err   = self._is_error(response)
        except Exception as exc:
            response = ""
            is_err   = True

        self._stats[version]["calls"]  += 1
        self._stats[version]["errors"] += int(is_err)

        self._check_rollback()                            # <3>

        return {"response": response, "version": version, "error": is_err}

    def _error_rate(self, version: str) -> float:
        s = self._stats[version]
        return s["errors"] / s["calls"] if s["calls"] > 0 else 0.0

    def _check_rollback(self) -> None:
        """Trigger rollback if canary error rate exceeds threshold."""
        if self._rolled_back:
            return
        c_rate = self._error_rate("canary")
        b_rate = self._error_rate("baseline")
        if self._stats["canary"]["calls"] < 10:
            return                                        # <4>
        if c_rate > self._rollback_mult * max(b_rate, 0.01):
            self.p = 0.0
            self._rolled_back = True
            msg = (
                f"[CANARY ROLLBACK] canary_err={c_rate:.2%}  "
                f"baseline_err={b_rate:.2%}  "
                f"ratio={c_rate/max(b_rate, 0.01):.1f}x"
            )
            self._alerts.append(msg)

    def status(self) -> dict:
        return {
            "p": self.p,
            "rolled_back": self._rolled_back,
            "baseline_error_rate": round(self._error_rate("baseline"), 4),
            "canary_error_rate":   round(self._error_rate("canary"),   4),
            "alerts": self._alerts,
            "stats": self._stats,
        }

1. For a compliance classifier, any response not in `{risk_factors, mda, capital_liquidity, guidance}` is a structural error. This provides a cheap, latency-free quality signal that does not require an LLM judge.
2. Rollback sets `p = 0`, so the `not self._rolled_back` guard short-circuits stochastic routing and routes 100% to baseline.
3. We check rollback after every request so that a misbehaving canary is caught within its first few dozen calls rather than waiting for a batch evaluation.
4. We require at least 10 canary calls before evaluating the rollback condition to avoid triggering on the first error (which could simply be statistical noise).

Simulating a canary that degrades after 15 calls:

In [ ]:
VALID_LABELS = {"risk_factors", "mda", "capital_liquidity", "guidance"}
QUERIES_CANARY = [
    "Our CET1 capital ratio was 14.8% at year-end.",
    "Net revenues increased 8% to $47.4 billion.",
    "VaR at the 99th percentile was $142 million.",
    "We expect EPS of $42–45 for fiscal 2025.",
    "Liquidity coverage ratio stands at 128%.",
]

# Simulate: baseline is gpt-4o-mini (production), canary is gpt-4o-mini (same, initially fine)
# We inject errors by monkey-patching the canary's complete method after 15 calls.

baseline_client = LLMClient(model="gpt-4o-mini")
canary_client   = LLMClient(model="gpt-4o-mini")

cr = CanaryRouter(
    baseline=baseline_client,
    canary=canary_client,
    p=0.50,  # High split so canary accumulates calls quickly in simulation
    valid_labels=VALID_LABELS,
)

call_log = []
canary_calls_so_far = 0

for i in range(80):
    # Inject a bad canary after it has handled 15 calls
    if canary_calls_so_far >= 15:
        # Overwrite complete to return garbage 60% of the time
        def _bad_complete(messages, *, response_format=None, _orig=canary_client.complete):
            if random.random() < 0.60:
                return "UNKNOWN_CATEGORY"
            return _orig(messages, response_format=response_format)
        canary_client.complete = _bad_complete

    query = QUERIES_CANARY[i % len(QUERIES_CANARY)]
    msgs  = [
        {"role": "system", "content": SYSTEM_CLASSIFY},
        {"role": "user",   "content": query},
    ]
    result = cr.route(msgs)
    if result["version"] == "canary":
        canary_calls_so_far += 1
    call_log.append(result)

    if cr._rolled_back:
        print(f"Rollback triggered at request {i+1}.")
        break

st = cr.status()
print(f"\nFinal status:")
print(f"  p={st['p']}  rolled_back={st['rolled_back']}")
print(f"  baseline error rate : {st['baseline_error_rate']:.2%}")
print(f"  canary error rate   : {st['canary_error_rate']:.2%}")
for alert in st["alerts"]:
    print(f"  {alert}")

## CI/CD Pipeline (GitHub Actions)

With the registry, drift detectors, and routers in place, we need a pipeline that orchestrates them automatically on every code change. The workflow below implements the full MLOps level-2 cycle: evaluation on every PR, Docker build and ECR push on merge to `main`, model registration, staging deployment with smoke tests, and a gated production promotion step.

Reference GitHub Actions workflow for a financial AI service:

In [ ]:
MLOPS_WORKFLOW = """\
name: MLOps — Compliance Classifier

on:
  pull_request:
    branches: [main]
  push:
    branches: [main]

env:
  AWS_REGION:    us-east-1
  ECR_REGISTRY:  123456789.dkr.ecr.us-east-1.amazonaws.com
  ECR_REPO:      compliance-classifier
  SERVICE_NAME:  compliance-classifier

jobs:
  # ── 1. Eval harness runs on every PR ────────────────────────────────────────
  eval:
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.13"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run eval harness
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: |
          python -m pytest evals/ -v --tb=short \
            --eval-threshold-accuracy=0.85 \
            --eval-threshold-f1=0.83

      - name: Upload eval report
        uses: actions/upload-artifact@v4
        with:
          name: eval-report
          path: evals/reports/

  # ── 2. Build and push Docker image on merge to main ─────────────────────────
  build-push:
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    outputs:
      image_tag: ${{ steps.meta.outputs.version }}
    steps:
      - uses: actions/checkout@v4

      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v4
        with:
          aws-access-key-id:     ${{ secrets.AWS_ACCESS_KEY_ID }}
          aws-secret-access-key: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          aws-region:            ${{ env.AWS_REGION }}

      - name: Login to ECR
        id: login-ecr
        uses: aws-actions/amazon-ecr-login@v2

      - name: Extract metadata
        id: meta
        uses: docker/metadata-action@v5
        with:
          images: ${{ env.ECR_REGISTRY }}/${{ env.ECR_REPO }}
          tags: |
            type=sha,prefix=,format=short
            type=raw,value=latest,enable=${{ github.ref == 'refs/heads/main' }}

      - name: Build and push
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags:   ${{ steps.meta.outputs.tags }}
          labels: ${{ steps.meta.outputs.labels }}
          cache-from: type=gha
          cache-to:   type=gha,mode=max

  # ── 3. Register model version in the registry ───────────────────────────────
  register:
    needs: build-push
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Register model version
        env:
          MODEL_REGISTRY_DB: ${{ secrets.MODEL_REGISTRY_DSN }}
        run: |
          python scripts/register_model.py \
            --name compliance-classifier \
            --version ${{ needs.build-push.outputs.image_tag }} \
            --base-model gpt-4o-mini \
            --artifact s3://models/compliance/${{ needs.build-push.outputs.image_tag }}/ \
            --metrics-file evals/reports/metrics.json

  # ── 4. Deploy to staging and run smoke tests ─────────────────────────────────
  staging:
    needs: register
    runs-on: ubuntu-latest
    environment: staging
    steps:
      - uses: actions/checkout@v4

      - name: Deploy to staging
        run: |
          kubectl set image deployment/${{ env.SERVICE_NAME }} \
            app=${{ env.ECR_REGISTRY }}/${{ env.ECR_REPO }}:${{ needs.build-push.outputs.image_tag }} \
            --namespace staging
          kubectl rollout status deployment/${{ env.SERVICE_NAME }} --namespace staging

      - name: Smoke tests
        run: |
          python scripts/smoke_test.py \
            --endpoint https://staging.compliance-ai.internal/classify \
            --queries evals/smoke_queries.json \
            --expected evals/smoke_expected.json

  # ── 5. Promote to production (requires manual approval) ────────────────────
  promote:
    needs: staging
    runs-on: ubuntu-latest
    environment: production          # GitHub environment with required reviewers
    steps:
      - uses: actions/checkout@v4

      - name: Promote model in registry
        env:
          MODEL_REGISTRY_DB: ${{ secrets.MODEL_REGISTRY_DSN }}
        run: |
          python scripts/promote_model.py \
            --name compliance-classifier \
            --version ${{ needs.build-push.outputs.image_tag }}

      - name: Deploy to production (canary at 5%)
        run: |
          python scripts/canary_deploy.py \
            --service ${{ env.SERVICE_NAME }} \
            --new-image ${{ env.ECR_REGISTRY }}/${{ env.ECR_REPO }}:${{ needs.build-push.outputs.image_tag }} \
            --canary-weight 5
"""
print(MLOPS_WORKFLOW)

The five jobs implement the full MLOps level-2 loop. A few design choices worth noting:

**Job 1 (eval).** The eval harness runs *only* on PRs, not on every push to `main`. This is deliberate: the `push` event on `main` triggers the build and deploy jobs which are sequential. Running evals again on `main` would be redundant since the PR already passed them, and would double the API cost.

**Job 3 (register).** The `scripts/register_model.py` script reads the `metrics.json` artifact uploaded by the eval job and inserts a new row in the SQLite registry (or MLflow if the team has scaled). The `image_tag` is the short Git SHA, providing a stable, human-readable link between a model version and its source commit.

**Job 5 (promote).** The `environment: production` key activates GitHub's environment protection rules. Any reviewer listed in the `production` environment's required-reviewers list must approve the deployment before the job runs. This is the human gate that satisfies financial-services change management requirements.

:::{.callout-caution}
The `MODEL_REGISTRY_DSN` secret must point to a persistent database (e.g., RDS PostgreSQL or an S3-backed DuckDB file), not the in-memory SQLite instance used in this notebook. An in-memory registry is destroyed at process exit and provides no audit trail between workflow runs.

:::

## Retraining Trigger

The final piece of the loop is the **retraining trigger**: a component that monitors the drift detectors and quality monitor, and fires a retraining event when either crosses its threshold. In production this event would invoke a SageMaker Pipelines execution, a GitHub Actions `workflow_dispatch`, or an Airflow DAG run. Here we implement the trigger logic and mock the invocation.

The trigger should be **rate-limited** to avoid thrashing: if drift is detected, we fire once and then enter a cooldown period (e.g., 24 hours) during which repeated drift signals do not launch additional retraining runs.

Implementing `RetrainingTrigger`:

In [ ]:
import time


class RetrainingTrigger:
    """
    Monitors drift and quality monitors; fires a retraining event
    when either signals alert, subject to a cooldown period.
    """

    def __init__(
        self,
        drift_detector: EmbeddingDriftDetector,
        quality_monitor: QualityDriftMonitor,
        cooldown_seconds: float = 86_400.0,  # 24 hours
    ):
        self._drift   = drift_detector
        self._quality = quality_monitor
        self._cooldown = cooldown_seconds
        self._last_trigger: Optional[float] = None   # <1>
        self._event_log: list[dict] = []

    def _in_cooldown(self) -> bool:
        if self._last_trigger is None:
            return False
        return (time.time() - self._last_trigger) < self._cooldown

    def _fire(self, reason: str, details: dict) -> dict:
        """Mock retraining invocation — in prod, calls SageMaker or workflow_dispatch."""
        self._last_trigger = time.time()
        event = {
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "reason": reason,
            "details": details,
            "action": "retraining_triggered",
        }
        self._event_log.append(event)
        print(f"[RetrainingTrigger] Firing: {reason}  →  {details}")
        return event

    def check(
        self,
        query_window: Optional[np.ndarray] = None,
        faithfulness: Optional[float] = None,
        relevancy: Optional[float] = None,
    ) -> Optional[dict]:
        """
        Evaluate current signals. Pass a query window for drift check and/or
        quality scores for quality check. Returns fired event or None.
        """
        if self._in_cooldown():                          # <2>
            return None

        # Check embedding drift
        if query_window is not None:
            drift_result = self._drift.check(query_window)
            if drift_result["alert"]:
                return self._fire(
                    reason="embedding_drift",
                    details={"mmd2": round(drift_result["mmd2"], 5),
                             "threshold": self._drift.threshold},
                )

        # Check output quality
        if faithfulness is not None and relevancy is not None:
            quality_state = self._quality.update(faithfulness, relevancy)
            if quality_state["alert"]:
                return self._fire(
                    reason="quality_drift",
                    details={
                        "ewma_faithfulness": quality_state["ewma_faithfulness"],
                        "ewma_relevancy":    quality_state["ewma_relevancy"],
                        "threshold": self._quality.threshold,
                    },
                )                                        # <3>

        return None

    def get_event_log(self) -> list[dict]:
        return list(self._event_log)

1. `_last_trigger` stores the Unix timestamp of the most recent retraining event. `None` means no event has ever been fired.
2. The cooldown gate is checked first — before running any expensive detector computation.
3. We check embedding drift before quality drift because embedding drift is a leading indicator: it often precedes quality degradation by several minutes in production. Triggering early on embedding drift may prevent the quality alert from ever firing.

Running a mock monitoring loop that fires on both drift types:

In [ ]:
rng3 = np.random.default_rng(99)

ref_embeddings = rng3.normal(0.0, 1.0, (200, 32))
fresh_detector = EmbeddingDriftDetector(reference=ref_embeddings, threshold=0.05)
fresh_monitor  = QualityDriftMonitor(span=20, threshold=0.70)

trigger = RetrainingTrigger(
    drift_detector=fresh_detector,
    quality_monitor=fresh_monitor,
    cooldown_seconds=0.0,  # No cooldown for demo purposes
)

print("=== Phase 1: No drift ===")
for _ in range(5):
    window = rng3.normal(0.0, 1.0, (100, 32))  # Same distribution
    event  = trigger.check(
        query_window=window,
        faithfulness=float(rng3.normal(0.88, 0.04)),
        relevancy=float(rng3.normal(0.86, 0.04)),
    )
    if event:
        print(f"  Event: {event}")

print("No alerts fired.\n")

print("=== Phase 2: Embedding drift ===")
drifted_window = rng3.normal(4.0, 1.0, (100, 32))  # Large shift
event = trigger.check(query_window=drifted_window)

print("\n=== Phase 3: Quality drift (simulated) ===")
for i in range(30):
    event = trigger.check(
        query_window=rng3.normal(0.0, 1.0, (100, 32)),
        faithfulness=float(rng3.normal(0.58, 0.07)),  # Degraded
        relevancy=float(rng3.normal(0.56, 0.08)),
    )
    if event:
        break

print(f"\nTotal retraining events fired: {len(trigger.get_event_log())}")
for ev in trigger.get_event_log():
    print(f"  [{ev['timestamp']}]  reason={ev['reason']}  details={ev['details']}")

:::{.callout-tip}
In a production SageMaker setup, `_fire()` would call `sagemaker.workflow.pipeline.Pipeline.start()` with the training dataset URI and new prompt hash as pipeline parameters. The pipeline handles data validation, training, offline evaluation, and model registration — all as auditable pipeline steps with logged inputs and outputs.

:::

## Appendix: Putting It Together {#sec-putting-together}

The components developed in this notebook form a closed loop. @fig-mlops-loop shows how they connect in a deployed financial AI system.

![(**a**) The three axes of change — code, data, model — each require their own change-detection mechanism. (**b**) The full MLOps loop: traffic enters the canary router; responses are logged to the drift detectors and quality monitor; the retraining trigger fires when thresholds are breached; the CI/CD pipeline retrains, validates, registers, and re-deploys.](./img/05-mlops-loop.png){#fig-mlops-loop width=90%}

**Key wiring decisions.**

- The `EmbeddingDriftDetector` should receive embeddings from the *retrieval* step, not the final answer generation step. This makes it sensitive to shifts in the query distribution before they propagate into output quality.
- The `QualityDriftMonitor` scores should be computed by a lightweight LLM judge (e.g., `gpt-4o-mini` scoring on a scale of 0–1) rather than by the production model itself — the production model cannot reliably self-evaluate.
- The `CanaryRouter` and `ShadowRouter` share the same `ModelRegistry`: both look up the current `production` version via `get_production()`. A registry that is not updated atomically during a promotion could cause the two routers to serve different versions.
- The `RetrainingTrigger`'s cooldown should be set to match the expected retraining duration plus a buffer. Firing a new retraining while a previous one is still in progress creates a version race that corrupts the registry audit trail.

<br>

**Regulatory context.** For a financial institution subject to SR 11-7 (model risk management guidance), every component above maps to a required control:

| Component | SR 11-7 control |
|:---|:---|
| `ModelRegistry` | Model inventory and version control |
| `EmbeddingDriftDetector` | Ongoing monitoring — input distribution |
| `QualityDriftMonitor` | Ongoing monitoring — output quality |
| `ShadowRouter` | Model validation (parallel run testing) |
| `CanaryRouter` | Controlled deployment with rollback |
| GitHub Actions workflow | Change management and audit trail |
| `RetrainingTrigger` | Periodic review and model updating |

---

$\blacksquare$